# Gaussian Process Inference

In this section, we will show how to perform posterior inference and make predictions using the GP priors we introduced in the last section. We will start with regression, where we can perform inference in closed form.

This is a “GPs in a nutshell” section to quickly get up and running with Gaussian processes in practice

. We’ll start coding all the basic operations from scratch, and then introduce GPyTorch, which will make working with state-of-the-art Gaussian processes and integration with deep neural networks much more convenient. We will consider these more advanced topics in depth in the next section. In that section, we will also consider settings where approximate inference is required — classification, point processes, or any non-Gaussian likelihoods.

## Posterior Inference for Regression

An observation model relates the function we want to learn, $f(x)$, to our observations $y(x)$, both indexed by some input $x$. 

- In classification, $x$ could be the pixels of an image, and $y$ could be the associated class label.

- In regression, $y$ typically represents a continuous output, such as a land surface temperature, a sea-level, a C02 concentration, etc.

In regression, we often assume the outputs are given by a latent noise-free function $f(x)$ plus i.i.d. Gaussian noise $\epsilon(x)$:
$$
y(x) = f(x) + \epsilon(x)
$$

with $\epsilon(x) \approx N(0,\sigma^2)$. Let $y = y(X) = (y(x_1),...,y(x_n))^T$ be a vector of our training observations,and $f = (f(x_1),...,f(x_n))^T$ be a vector of the latent noise-free function values, queried at the training inputs $X = x_1,...,x_n$.

We will assume $f(x) \approx GP(m,k)$, which means that any collection of function values $f$ has a joint multivariate Gaussian distribution, with mean vector $\mu_i = m(x_i)$ and covariance matrix $K_{ik} = k(x_i,x_j)$.

The RBF kernel:
$$
k(x_i,x_j) = a^2.\exp(-\frac{1}{2.l^2}||x_i - x_j||^2)
$$

would be a standard choice of covariance function.

For notational simplicity, we will assume the mean function $m(x) = 0$, our derivations can easily be generalized later on.

Suppose we want to make predictions at a set of inputs:
$$
X_* = x_{*1},x_{*2},...,x_{*m}
$$

Then we want to find $x^2$ and $p(f_*|y, X)$. In the regression setting, we can conveniently find this distribution by using Gaussian identities, after finding the joint distribution over $f_{*} = f(X_*)$ and $y$.

If we evaluate equation (18.3.1) at the training inputs $X$, we have $y = f + \epsilon$. By the definition of a Gaussian process (see last section), $f \approx N(0, K(X,X))$ where $K(X, X)$ is an $n \times n$  matrix formed by evaluating our covariance function (aka kernel) at all possible pairs of inputs $x_i,x_j \in X$.

$\epsilon$ is simply a vector comprised of iid samples from $N(0,\sigma^2)$ and thus has distribution $N(0,\sigma^2.I)$. $y$ is therefore a sum of two independent multivariate Gaussian variables, and thus has distribution $N(0, K(X,X) + \sigma^2.I)$.

One can also show that:
$$
cov(f_*,y) = cov(y,f_*)^T = K(X_*,X)
$$

whre:
- $K(X_*,X)$ is an $m \times n$ matrix formed by evaluating the kernel at all pairs of test and training inputs.

$$
\begin{bmatrix}
y \\ f_{*}
\end{bmatrix} \approx N(0, A = \begin{bmatrix}
K(X,X) + \sigma^2.I && K(X,X_*) \\
K(X_*,X) && K(X_*,X_*)
\end{bmatrix})
$$

We can then use standard Gaussian identities to find the conditional distribution from the joint distribution (see, e.g., Bishop Chapter 2):
$$
f_*|y,X,X_* \approx N(m_*,S_*)
$$

where:
- $m_* = K(X_*,X)[K(X,X) + \sigma^2.I]^{-1}.y$ 
- $S_* = K(X_*,X_*) - K(X_*,X)[K(X,X) + \sigma^2.I]^{-1}.K(X,X_*)$

Typically, we do not need to make use of the full predictive covariance matrix $S$, and instead use the diagonal of $S$  for uncertainty about each prediction. Often for this reason we write the predictive distribution for a single test point $x_*$ rather than a collection of test points

The kernel matrix has parameters $\theta$ hat we also wish to estimate, such the amplitude $a$ and lengthscale $l$ of the RBF kernel above. 

For these purposes we use the marginal likelihood, $p(y|\theta,X)$ which we already derived in working out the marginal distributions to find the joint distribution over $y,f_*$.

As we will see, the marginal likelihood compartmentalizes into model fit and model complexity terms, and automatically encodes a notion of Occam’s razor for learning hyperparameters

For a full discussion, see MacKay Ch. 28 (MacKay, 2003), and Rasmussen and Williams Ch. 5 (Rasmussen and Williams, 2006).

In [ ]:
import math
import os
import gpytorch
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import optimize
from scipy.spatial import distance_matrix
from d2l import torch as d2l

d2l.set_figsize()

## Equations for Making Predictions and Learning Kernel Hyperparameters in GP Regression

We list here the equations you will use for learning hyperparameters and making predictions in Gaussian process regression.

Again, we assume a vector of regression targets $y$, indexed by inputs $X = {x_1,...,x_n}$, and we wish to make a prediction at a test input $x_*$.

We assume i.i.d. additive zero-mean Gaussian noise with variance $\sigma^2$. We use a Gaussian process prior $f(x) \approx GP(m,k)$ for the latent noise-free function, with mean function $m$ and kernel function $k$.

The kernel itself has parameters $\theta$ that we want to learn.

For example, if we use an RBF kernel:
$$
k(x_i,x_j) = a^2.\exp(-\frac{1}{2l^2}||x - x^{'}||^2)
$$

we want to learn $\theta = \{a^2, l^2\}$

Let $K(X,X)$ represent an $n \times n$ matrix corresponding to evaluating the kernel for all possible pairs of $n$ training inputs. Let $K(x_*,X)$ represent a $1 \times n$ vector formed by evaluating $k(x_*,x_i), i=1,...,n$.

Let $\mu$ be a mean vector formed by evaluating the mean function $m(x)$ at every training points $x$.

Typically in working with Gaussian processes, we follow a two-step procedure:

1. Learn kernel hyperparameters $\hat{\theta}$ by maximizing the marginal likelihood with respect to these hyperparameters

2. Use the predictive mean as a point predictor, and 2 times the predictive standard deviation to form a 95% credible set, conditioning on these learned hyperparameters $\hat{\theta}$

The log marginal likelihood is simply a log Gaussian density, which has the form:

$$
\log(p(y|\theta,X)) = -\frac{1}{2}.y^T.[K_\theta(X,X)+\sigma^2.I]^{-1}.y - \frac{1}{2}.\log(|K_\theta(X,X)|)+c
$$

The predictive distribution has the form:

$$
p(y_*|x_*,y,\theta) = N(a_*,v_*)
$$

$$
a_* = k_\theta(x_*,X).[K_\theta(X,X) + \sigma^2.I]^{-1}.(y - \mu) + \mu
$$

$$
v_* = k_\theta(x_*,x_*) - K_\theta(x_*,X)[K_\theta(X,X) + \sigma^2.I]^{-1}.k_\theta(X,x_*)
$$

## Interpreting Equations for Learning and Predictions

There are some key points to note about the predictive distributions for Gaussian processes:

- Despite the flexibility of the model class, it is possible to do exact Bayesian inference for GP regression in closed form. Aside from learning the kernel hyperparameters, there is no training. We can write down exactly what equations we want to use to make predictions. Gaussian processes are relatively exceptional in this respect, and it has greatly contributed to their convenience, versatility, and continued popularity.

- The predictive mean $a_*$ is a linear combination of the training targets $y$, weighted by the kernel $k_\theta(x_*,X)[K_\theta(X,X) + \sigma^2.I]^{-1}$. As we will see, the kernel (and its hyperparameters) thus plays a crucial role in the generalization properties of the model.

- The predictive mean explicitly depends on the target values $y$ but the predictive variance does not. The predictive uncertainty instead grows as the test input $x_*$ moves away from the target locations $X$, as governed by the kernel function. However, uncertainty will implicitly depend on the values of the targets $y$ through the kernel hyperparameters $\theta$ which are learned from the data.

- The marginal likelihood compartmentalizes into model fit and model complexity (log determinant) terms. The marginal likelihood tends to select for hyperparameters that provide the simplest fits that are still consistent with the data.

- The key computational bottlenecks come from solving a linear system and computing a log determinant over an $n \times n$ symmetric positive definite matrix $K(X,X)$ for $n$ training points. Naively, these operations each incur $O(n^3)$ computations, as well as $O(n^2)$ storage for each entry of the kernel (covariance) matrix, often starting with a Cholesky decomposition. Historically, these bottlenecks have limited GPs to problems with fewer than about 10,000 training points, and have given GPs a reputation for “being slow” that has been inaccurate now for almost a decade. In advanced topics, we will discuss how GPs can be scaled to problems with millions of points.

- For popular choices of kernel functions, $K(X, X)$ is often close to singular, which can cause numerical issues when performing Cholesky decompositions or other operations intended to solve linear systems. Fortunately, in regression we are often working with $K_\theta(X,X) + \sigma^2.I$, such that the noise variance $\sigma^2$ gets added to the diagonal of $K(X,X)$, significantly improving its conditioning. If the noise variance is small, or we are doing noise free regression, it is common practice to add a small amount of “jitter” to the diagonal, on the order of $10^{-6}$, to improve conditioning.

## Worked example from scratch

Let’s create some regression data, and then fit the data with a GP, implementing every step from scratch. We’ll sample data from:
$$
y(x) = \sin(x) + \frac{1}{2}.\sin(4x) + \epsilon
$$

with $\epsilon \approx N(0,\sigma^2)$. The noise free function we wish to find is:
$$
f(x) = \sin(x) + \frac{1}{2}.\sin(4x)
$$

We’ll start by using a noise standard deviation $\sigma = 0.25$.

In [ ]:
def data_maker1(x, sig):
    return np.sin(x) + 0.5.np.sin(4*x) + np.random.randn(x.shape[0]) * sig

sig = 0.25
train_x, test_x = np.linspace(0, 5, 50), np.linspace(0, 5, 500)
train_y, test_y = data_maker1(train_x, sig=sig), data_maker1(test_x, sig=0.)

d2l.plt.scatter(train_x, train_y)
d2l.plt.plot(test_x, test_y)
d2l.plt.xlabel("x", fontsize=20)
d2l.plt.ylabel("Observations y", fontsize=20)
d2l.plt.show()

![](image1.png)

Here we see the noisy observations as circles, and the noise-free function in blue that we wish to find.

Now, let’s specify a GP prior over the latent noise-free function, $f(x) \approx GP(m,k)$. We’ll use a mean function $m(x) = 0$, and an RBF covariance function (kernel):
$$
k(x_i,x_j) = a^2.\exp(-\frac{1}{2.l^2}||x - x^{'}||^2)
$$

In [ ]:
mean = np.zeros(test_x.shape[0])
cov = d2l.rbfkernel(test_x, test_x, ls=0.2)

We have started with a length-scale of 0.2. Before we fit the data, it is important to consider whether we have specified a reasonable prior

Let’s visualize some sample functions from this prior, as well as the 95% credible set (we believe there’s a 95% chance that the true function is within this region).

In [ ]:
prior_samples = np.random.multivariate_normal(mean=mean, cov=cov, size=5)
d2l.plt.plot(test_x, prior_samples.T, color='black', alpha=0.5)
d2l.plt.plot(test_x, mean, linewidth=2.)
d2l.plt.fill_between(
    test_x,
    mean - 2 * np.diag(cov),
    mean + 2 * np.diag(cov),
    alpha=0.25
)
d2l.plt.show()

![](image2.png)

Do these samples look reasonable? Are the high-level properties of the functions aligned with the type of data we are trying to model?

Now let’s form the mean and variance of the posterior predictive distribution at any arbitrary test point $x_{*}$.

$$
f_* = K(x,x_*)^T.(K(x,x) + \sigma^2.I)^{-1}.y
$$
$$
V(f_*) = K(x_*,x_*) - K(x,x_*)^T.(K(x,x) + \sigma^2.I)^{-1}.K(x,x_*)
$$

Before we make predictions, we should learn our kernel hyperparameters $\theta$ and noise variance $\sigma^2$.

Let’s initialize our length-scale at 0.75, as our prior functions looked too quickly varying compared to the data we are fitting. We’ll also guess a noise standard deviation $\sigma$ of 0.75.

In order to learn these parameters, we will maximize the marginal likelihood with respect to these parameters:
$$
\log{p(y|X)} = \log{\int{p(y|f,X).p(f|X)df}}
$$

$$\log{p(y|X)} = -\frac{1}{2}.y^T.(K(x,x) + \sigma^2.I)^{-1}.y - \frac{1}{2}.\log{|K(x,x) + \sigma^2.I|} - \frac{n}{2}.\log{2\pi}$$

Perhaps our prior functions were too quickly varying. Let’s guess a length-scale of 0.4. We’ll also guess a noise standard deviation of 0.75. These are simply hyperparameter initializations — we will learn these parameters from the marginal likelihood.

In [ ]:
ell_est = 0.4
post_sig_est = 0.5

def neg_MLL(pars):
    K = d2l.rbfkernel(train_x, train_x, ls=pars[0])
    kernel_term = -0.5 * train_y @ np.linalg.inv(K + pars[1] ** 2 * np.eye(train_x.shape[0])) @ train_y
    logdet = -0.5 * np.log(np.linalg.det(K + pars[1] ** 2 * np.eye(train_x.shape[0])))
    const = -train_x.shape[0] / 2. * np.log(2 * np.pi)

    return -(kernel_term + logdet + const)
    

learned_hypers = optimize.minimize(
    neg_MLL,
    x0=np.array([
        ell_est,
        post_sig_est
    ]),
    bounds=((0.01, 10.0), (0.01, 10.))
)
ell = learned_hypers.x[0]
post_sig_est = learned_hypers.x[1]